## **Cleaning and EDA**

In [42]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

import build_dataset as bd
import importlib
importlib.reload(bd)

from build_dataset import DATA_DIR, FILE_ORDER, TARGET, LABEL_MAP

pd.set_option("display.max_colwidth", None)

**1. Schema Verification**

The CIC-IDS2017 dataset captures network traffic data over a week (July 3-7 2017). Data is split into 8 csv files based on day of week and attack type, but was captured continuously with the same methodology, making merging valid.

Build out functions used in this notebook can be found in `../src/build_dataset.py `.

In [40]:
# VERIFY SCHEMAS OF INDIVIDUAL DATA FILES
schemas = bd.read_schemas()
print(f"{bd.n_distinct_schemas(schemas)} distinct schema(s) across data files.")

1 distinct schema(s) across data files.


**2. Within file cleaning then merge separate files**

`merge_files` cleans the data for each day before appending to a single parquet file. 
- Strips whitepace and drops duplicate columns
- Repairs mojibake dash in `Web Attack` labels
- Drops rows with nulls with null values in numeric features and within-file duplicates (REVISIT VALIDITY)
- Downcasts numerics to float 32

In [41]:
# CLEAN AND MERGE INDIVIDUAL DATA FILES
merged_path, per_file_counts = bd.merge_files()
print("\nMerged to path:", merged_path)

Monday-WorkingHours.pcap_ISCX.csv: 502,650 rows
Tuesday-WorkingHours.pcap_ISCX.csv: 421,626 rows
Wednesday-workingHours.pcap_ISCX.csv: 610,492 rows
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 164,179 rows
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 252,790 rows
Friday-WorkingHours-Morning.pcap_ISCX.csv: 184,044 rows
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 213,777 rows
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 223,082 rows

Merged to path: c:\Users\lixin\portfolio\cybersecurity_project\data\merged_rawlabels.parquet


**3. Cross-file cleaning, constant column removal, and column renaming**

`clean_merged` passes for cross-file duplicates and constant columns, then converts column names to camel case. 

In [43]:
# FILTER MERGED DATA
stats = bd.clean_merged()
print("Constant columns:", stats["constant_cols"])
print("\nWritten to path:", stats["path"])

Number of inter-file duplicates: 74,660
Number of constant columns: 8
Rows written: 2,497,980
Constant columns: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Written to path: c:\Users\lixin\portfolio\cybersecurity_project\data\merge_complete.parquet


In [44]:
# CONFIRM FINAL COLUMN NAMES
print(stats["columns"])

['DestinationPort', 'FlowDuration', 'TotalFwdPackets', 'TotalBackwardPackets', 'TotalLengthOfFwdPackets', 'TotalLengthOfBwdPackets', 'FwdPacketLengthMax', 'FwdPacketLengthMin', 'FwdPacketLengthMean', 'FwdPacketLengthStd', 'BwdPacketLengthMax', 'BwdPacketLengthMin', 'BwdPacketLengthMean', 'BwdPacketLengthStd', 'FlowBytesS', 'FlowPacketsS', 'FlowIatMean', 'FlowIatStd', 'FlowIatMax', 'FlowIatMin', 'FwdIatTotal', 'FwdIatMean', 'FwdIatStd', 'FwdIatMax', 'FwdIatMin', 'BwdIatTotal', 'BwdIatMean', 'BwdIatStd', 'BwdIatMax', 'BwdIatMin', 'FwdPshFlags', 'FwdUrgFlags', 'FwdHeaderLength', 'BwdHeaderLength', 'FwdPacketsS', 'BwdPacketsS', 'MinPacketLength', 'MaxPacketLength', 'PacketLengthMean', 'PacketLengthStd', 'PacketLengthVariance', 'FinFlagCount', 'SynFlagCount', 'RstFlagCount', 'PshFlagCount', 'AckFlagCount', 'UrgFlagCount', 'CweFlagCount', 'EceFlagCount', 'DownUpRatio', 'AveragePacketSize', 'AvgFwdSegmentSize', 'AvgBwdSegmentSize', 'SubflowFwdPackets', 'SubflowFwdBytes', 'SubflowBwdPackets', 

**4. Understanding attack types present in dataset**

`label_by_file` lists the unique attack types and the number of rows present in each original data file. 
`label_distribution`

In [45]:
# DETERMINE ATTACK TYPES APPEARING IN EACH FILE WITH COUNTS
bd.label_by_file()

,file,n_rows,attack_types
0,Monday-WorkingHours.pcap_ISCX.csv,502573,BENIGN
1,Tuesday-WorkingHours.pcap_ISCX.csv,414134,"BENIGN, FTP-Patator, SSH-Patator"
2,Wednesday-workingHours.pcap_ISCX.csv,594193,"BENIGN, DoS GoldenEye, DoS Hulk, DoS Slowhttptest, DoS slowloris, Heartbleed"
3,Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv,154104,"BENIGN, Web Attack - Brute Force, Web Attack - Sql Injection, Web Attack - XSS"
4,Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv,240082,"BENIGN, Infiltration"
5,Friday-WorkingHours-Morning.pcap_ISCX.csv,171359,"BENIGN, Bot"
6,Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,204230,"BENIGN, PortScan"
7,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,217305,"BENIGN, DDoS"


In [46]:
# DETERMINE FREQUENCY OF EACH LABEL
bd.label_distribution()

,Label,Count,Percentage
0,BENIGN,2072254,82.96
1,DoS Hulk,172846,6.92
2,DDoS,128014,5.12
3,PortScan,90694,3.63
4,DoS GoldenEye,10282,0.41
5,FTP-Patator,5931,0.24
6,DoS slowloris,5374,0.22
7,DoS Slowhttptest,5228,0.21
8,SSH-Patator,3219,0.13
9,Bot,1948,0.08


Labels representing similar attacks are grouped together. This is because some classes are too small to evaluate on their own, and several labels represent the same type of attack. Infiltration and Heartbleed are excluded due to lack of data. 

1. DDoS
2. DoS
    DoS Hulk, DoS Golden Eye, DoS slowloris, DoS Slowhttptest
3. PortScan
4. Bot
5. BruteForce
    FTP-Patator, SSH-Patator
6. WebAttack
    Web Attack - Brute Force, Web Attack - XSS, Web Attack - Sql Injection
7. EXCLUDED
    Infiltration [36 attacks], Heartbleed [11 attacks]

In [47]:
# MAP LABELS TO GROUPS
LABEL_MAP

{'BENIGN': 'BENIGN',
 'DDoS': 'DDoS',
 'DoS Hulk': 'DoS',
 'DoS GoldenEye': 'DoS',
 'DoS slowloris': 'DoS',
 'DoS Slowhttptest': 'DoS',
 'PortScan': 'PortScan',
 'Bot': 'Bot',
 'FTP-Patator': 'BruteForce',
 'SSH-Patator': 'BruteForce',
 'Web Attack - Brute Force': 'WebAttack',
 'Web Attack - XSS': 'WebAttack',
 'Web Attack - Sql Injection': 'WebAttack',
 'Infiltration': 'EXCLUDED',
 'Heartbleed': 'EXCLUDED'}

In [48]:
df = bd.load_dataset(columns=[TARGET, "SourceFile"], label_group=True, drop_excluded=True)
print(df["LabelGroup"].value_counts(dropna=False))

LabelGroup
BENIGN        2072254
DoS            193730
DDoS           128014
PortScan        90694
BruteForce       9150
WebAttack        2143
Bot              1948
Name: count, dtype: int64
